In [1]:
##importing liibraries
from binance import Client
from binance.enums import HistoricalKlinesType
import pandas as pd
import math
from pandas import Series, DataFrame
from pandas_ta.overlap import hlc3
from pandas_ta.utils import get_offset, is_datetime_ordered, verify_series
import numpy as np
%run ./binance_keys.ipynb

##api key
client = Client(api_key,api_secret)

##inputs
symbol = 'RUNEUSDT'
interval = '15m'
bands_dev = [-2,2]
qty = 10

def getminutedata(symbol, interval, lookback):
    
    #retreiving data
    historical_data = client.get_historical_klines(symbol, interval,lookback + 'min ago UTC', klines_type=HistoricalKlinesType.FUTURES)

    # Create a DataFrame from the fetched data
    df = pd.DataFrame(historical_data)
    
    # Select the first 6 columns
    df = df.iloc[:, :6]
    
    # Rename columns
    df.columns = ["open_time", "open", "high", "low", "close", "volume"]
    
    # Convert the 'open_time' column to a datetime format and set as index
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df = df.set_index(df.open_time)
    df = df.drop(['open_time'], axis=1)
    df = df.astype(float)
    
    return df

# calculate vwap with bands
def vwap_with_bands(
    high: Series, low: Series, close: Series, volume: Series, bands: list = [-1,1],
    anchor: str = None,
    offset: int = None, **kwargs
):
   
    # Validate
    high = verify_series(high)
    low = verify_series(low)
    close = verify_series(close)
    volume = verify_series(volume)
    anchor = anchor.upper() if anchor and isinstance(
        anchor, str) and len(anchor) >= 1 else "D"
    offset = get_offset(offset)

    typical_price = hlc3(high=high, low=low, close=close)
    if not is_datetime_ordered(volume):
        _s = "[!] VWAP volume series is not datetime ordered."
        print(f"{_s} Results may not be as expected.")
    if not is_datetime_ordered(typical_price):
        _s = "[!] VWAP price series is not datetime ordered."
        print(f"{_s} Results may not be as expected.")

    # Calculate vwap
    wp = typical_price * volume
    vwap = wp.groupby(wp.index.to_period(anchor)).cumsum()
    vwap /= volume.groupby(volume.index.to_period(anchor)).cumsum()

    # Calculate vwap stdev bands
    var = volume * (typical_price *typical_price)
    var_sum = var.groupby(var.index.to_period(anchor)).cumsum()
    volume_sum = volume.groupby(volume.index.to_period(anchor)).cumsum()
    stddev_vwap = abs(var_sum/volume_sum - vwap*vwap)
    std_volume_weighted =  stddev_vwap.apply(math.sqrt)
    
    # Build Dataframe
    df = pd.DataFrame()
    df[f"VWAP_{anchor}"] = vwap
    for i in bands:
        df[f"{i}_VWAP_band"] = vwap + (i*std_volume_weighted)

    # Offset
    if offset != 0:
        vwap = vwap.shift(offset)

    # Fill
    if "fillna" in kwargs:
        vwap.fillna(kwargs["fillna"], inplace=True)
    if "fill_method" in kwargs:
        vwap.fillna(method=kwargs["fill_method"], inplace=True)

    # Name and Category
    vwap.name = f"VWAP_{anchor}_bands"
    vwap.category = "overlap"

    return df

#find signals 
def find_signal(close, high, low, vwap,  upperband, lowerband):
    if (low < lowerband) & (close > lowerband) & (close < vwap):
        return 'buy'
    elif (high > upperband) & (close < upperband) & (close > vwap):
        return 'sell'


def getdata_withbands():
    #getting minute data
    df =getminutedata(symbol, interval, '1440')
        
    # Calculate the VWAP with bands
    vwap_bands_df = vwap_with_bands(
        high=df["high"],
        low=df["low"],
        close=df["close"],
        volume=df["volume"],
        bands=bands_dev,
        anchor="D")
        
    ##merge vwap with bands with og df
    new = pd.merge(df, vwap_bands_df, left_index=True, right_index=True)
    new["open_time"] = df.index
    new['signal'] = np.vectorize(find_signal)(df['close'],df['high'], df['low'],new['VWAP_D'], new.iloc[:, 7], new.iloc[:, 6])
    return new

In [2]:
def tradingstrat(open_position=False):
    while True:
        data = getdata_withbands()
        
        if not open_position:
            if data['signal'].iloc[-2] == "buy":
                print("BUY!")
                open_position = True
            else:
                print("no trade")
                print(data.close.iloc[-1])
        
        if open_position:  # Check for conditions to close the position only when in trade
            if (data['close'].iloc[-1] >= data['VWAP_D'].iloc[-1]) and (not data.empty):
                print("BUY POSITION SQUARED OFF!")
                open_position = False
            else:
                print("in trade")
                print(data.close.iloc[-1])
                

In [3]:
getdata_withbands()

,open,high,low,close,volume,VWAP_D,-2_VWAP_band,2_VWAP_band,open_time,signal
open_time,,,,,,,,,,
2023-09-15 22:00:00,1.771,1.780,1.764,1.778,1150975.0,1.774000,1.774000,1.774000,2023-09-15 22:00:00,None
2023-09-15 22:15:00,1.778,1.808,1.777,1.805,2862845.0,1.790167,1.769665,1.810669,2023-09-15 22:15:00,None
2023-09-15 22:30:00,1.805,1.844,1.796,1.818,5842487.0,1.807456,1.775951,1.838961,2023-09-15 22:30:00,sell
2023-09-15 22:45:00,1.819,1.828,1.811,1.821,2053118.0,1.809618,1.779431,1.839805,2023-09-15 22:45:00,None
2023-09-15 23:00:00,1.820,1.837,1.820,1.828,1607237.0,1.811844,1.781027,1.842661,2023-09-15 23:00:00,None
...,...,...,...,...,...,...,...,...,...,...
2023-09-16 20:45:00,1.913,1.914,1.906,1.910,619674.0,1.889312,1.842578,1.936046,2023-09-16 20:45:00,None
2023-09-16 21:00:00,1.910,1.914,1.908,1.911,466812.0,1.889424,1.842707,1.936140,2023-09-16 21:00:00,None
2023-09-16 21:15:00,1.911,1.924,1.911,1.919,881743.0,1.889700,1.842875,1.936525,2023-09-16 21:15:00,None


In [ ]:
while True:
    tradingstrat(open_position = False)

no trade
1.936
no trade
1.936
no trade
1.935
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.935
no trade
1.936
no trade
1.935
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.935
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.936
no trade
1.935
no trade
1.936
no trade
1